# RepFR Analyses Runner - LohnasKahana2014

Run repetition-focused free recall analyses via papermill.

**Paradigm:** Free Recall with item repetitions
**Reference:** @lohnas2014retrieved

In [ ]:
import os
import numpy as np
import papermill as pm

from jaxcmr.helpers import find_project_root, generate_trial_mask, load_data

## Dataset Configuration

In [ ]:
# ============================================================================
# DATASET CONFIGURATION - LohnasKahana2014
# ============================================================================
# Paradigm: Free Recall with spacing manipulation
# Reference: @lohnas2014retrieved
# List types: 1 (control), 3 (massed), 4 (spaced lag 1-8)
# ============================================================================

DATA_NAME = "RepeatedRecallsLohnasKahana2014"
DATA_FILE = "RepeatedRecallsLohnasKahana2014.h5"

# Trial queries (from thesis)
TRIAL_QUERY = "data['list_type'] > 0"              # All non-control lists
CONTROL_QUERY = "data['list_type'] == 1"           # Control lists only

# List type combinations for analysis (from thesis)
LT_COMBINATIONS = [
    ("4", [4]),       # Spaced only
    ("34", [3, 4]),   # Massed + spaced
]

# Analyses from thesis LohnasKahana2014.ipynb
SINGLE_ANALYSIS_PATHS = [
    "jaxcmr.analyses.repcrp.plot_rep_crp",
    "jaxcmr.analyses.backrepcrp.plot_back_rep_crp",
]

COMPARISON_ANALYSIS_PATHS = [
    "jaxcmr.analyses.spc.plot_spc",
    "jaxcmr.analyses.crp.plot_crp",
    "jaxcmr.analyses.pnr.plot_pnr",
    "jaxcmr.analyses.repneighborcrp.plot_repneighborcrp_i2j",
    "jaxcmr.analyses.repneighborcrp.plot_repneighborcrp_j2i",
    "jaxcmr.analyses.repneighborcrp.plot_repneighborcrp_both",
    "jaxcmr.analyses.rpl.plot_rpl",
    "jaxcmr.analyses.rpl.plot_full_rpl",
]

In [ ]:
project_root = find_project_root()
templates_dir = os.path.join(project_root, "templates")
rendered_dir = os.path.join(project_root, "projects/repfr/notebooks/rendered")
figures_dir = os.path.join(project_root, "projects/repfr/results/figures/analyses")

os.makedirs(rendered_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)

data_path = os.path.join(project_root, "data", DATA_FILE)
data = load_data(data_path)
print(f"Loaded data from {data_path}")

## Analysis Specifications

Following the thesis pattern from `work/thesis/LohnasKahana2014.ipynb`:
- **Single analyses**: repcrp, backrepcrp (for mixed and control separately)
- **Comparison analyses**: spc, crp, pnr, repneighborcrp variants, rpl (mixed vs control)

Analyses are run for each list type combination (LT4 = spaced only, LT34 = massed + spaced)

In [ ]:
# Build analysis specs following thesis pattern
analysis_specs = []

for lt_tag, lt_values in LT_COMBINATIONS:
    lt_query = f"np.isin(data['list_type'].flatten(), {lt_values})"
    
    # Basic benchmarks
    analysis_specs.append({
        "name": "spc",
        "template": "spc.ipynb",
        "parameters": {
            "run_tag": f"SPC_LT{lt_tag}",
            "data_name": DATA_NAME,
            "data_query": lt_query,
            "output_path": os.path.join(figures_dir, f"spc_{DATA_NAME}_LT{lt_tag}.png"),
        },
        "lt_tag": lt_tag,
    })
    
    analysis_specs.append({
        "name": "crp",
        "template": "crp.ipynb",
        "parameters": {
            "run_tag": f"CRP_LT{lt_tag}",
            "data_name": DATA_NAME,
            "data_query": lt_query,
            "output_path": os.path.join(figures_dir, f"crp_{DATA_NAME}_LT{lt_tag}.png"),
        },
        "lt_tag": lt_tag,
    })
    
    analysis_specs.append({
        "name": "pnr",
        "template": "pnr.ipynb",
        "parameters": {
            "run_tag": f"PNR_LT{lt_tag}",
            "data_name": DATA_NAME,
            "data_query": lt_query,
            "query_recall_position": 0,
            "output_path": os.path.join(figures_dir, f"pnr_{DATA_NAME}_LT{lt_tag}.png"),
        },
        "lt_tag": lt_tag,
    })
    
    # Repetition CRP
    analysis_specs.append({
        "name": "repcrp",
        "template": "repcrp.ipynb",
        "parameters": {
            "data_name": DATA_NAME,
            "data_query": lt_query,
            "ctrl_query": CONTROL_QUERY,
            "min_lag": 4,
            "output_path": os.path.join(figures_dir, f"repcrp_{DATA_NAME}_LT{lt_tag}.png"),
        },
        "lt_tag": lt_tag,
    })
    
    # Backward Repetition CRP
    analysis_specs.append({
        "name": "backrepcrp",
        "template": "backrepcrp.ipynb",
        "parameters": {
            "data_name": DATA_NAME,
            "data_query": lt_query,
            "ctrl_query": CONTROL_QUERY,
            "min_lag": 4,
            "output_path": os.path.join(figures_dir, f"backrepcrp_{DATA_NAME}_LT{lt_tag}.png"),
        },
        "lt_tag": lt_tag,
    })
    
    # Repetition Neighbor CRP (3 variants from thesis)
    analysis_specs.append({
        "name": "repneighborcrp",
        "template": "repneighborcrp.ipynb",
        "parameters": {
            "data_name": DATA_NAME,
            "data_query": lt_query,
            "ctrl_query": CONTROL_QUERY,
            "min_lag": 4,
            "output_path": os.path.join(figures_dir, f"repneighborcrp_{DATA_NAME}_LT{lt_tag}.png"),
        },
        "lt_tag": lt_tag,
    })
    
    # RPL (Recall Probability Learning / Spacing effect)
    analysis_specs.append({
        "name": "rpl",
        "template": "rpl.ipynb",
        "parameters": {
            "data_name": DATA_NAME,
            "data_query": lt_query,
            "output_path": os.path.join(figures_dir, f"rpl_{DATA_NAME}_LT{lt_tag}.png"),
        },
        "lt_tag": lt_tag,
    })

## Run Analyses

In [ ]:
for spec in analysis_specs:
    template_path = os.path.join(templates_dir, spec["template"])
    
    if not os.path.exists(template_path):
        print(f"WARNING: Template not found: {template_path}")
        continue
    
    lt_tag = spec.get("lt_tag", "")
    rendered_name = f"{spec['name']}_{DATA_NAME}_LT{lt_tag}.ipynb"
    output_path = os.path.join(rendered_dir, rendered_name)
    
    print(f"Running {spec['name']} (LT{lt_tag}) -> {output_path}")
    pm.execute_notebook(
        template_path,
        output_path,
        parameters=spec["parameters"],
    )